In [0]:
T_trans=spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

def to_dateid(dt):
    return int(dt.strftime('%Y%m%d'))

# Keep datetime
post_window_dt = datetime.strptime('2026-04-30', '%Y-%m-%d')

# Create PY datetime
PY_post_window_dt = post_window_dt - relativedelta(years=1)

# Convert both to dateid
post_window = to_dateid(post_window_dt)
PY_post_window = to_dateid(PY_post_window_dt)

In [0]:
tags = [
    'BP TAG+$10 - 52 CR',
    'BP TAG+$100 - 802 CR',
    'BP TAG+$129 - 1107 CR',
    'BP TAG+$15 - 72 CR',
    'BP TAG+$25 - 172 CR',
    'BP TAG+$35 - 257 CR',
    'BP TAG+$50 - 377 CR',
    'BP TAG+$75 - 577 CR',
    'PREM TAG+$10 - 54 CR',
    'PREM TAG+$100 - 804 CR',
    'PREM TAG+$129 - 1109 CR',
    'PREM TAG+$15 - 74 CR',
    'PREM TAG+$25 - 174 CR',
    'PREM TAG+$35 - 257 CR',
    'PREM TAG+$35 - 259 CR',
    'PREM TAG+$50 - 379 CR',
    'PREM TAG+$75 - 579 CR',
    'PROMO BP TAG $100 - 1002 CR',
    'TAG $10 - 40 CR',
    'TAG $100 - 790 CR',
    'TAG $129 - 1095 CR',
    'TAG $15 - 60 CR',
    'TAG $25 - 160 CR',
    'TAG $35 - 245 CR',
    'TAG $5 - 20 CR',
    'TAG $50 - 365 CR',
    'TAG $75 - 565 CR',
]

tag_bins = {
    '$5': [
        'TAG $5 - 20 CR',
    ],
    '$10': [
        'BP TAG+$10 - 52 CR',
        'PREM TAG+$10 - 54 CR',
        'TAG $10 - 40 CR',
    ],
    '$15': [
        'BP TAG+$15 - 72 CR',
        'PREM TAG+$15 - 74 CR',
        'TAG $15 - 60 CR',
    ],
    '$25': [
        'BP TAG+$25 - 172 CR',
        'PREM TAG+$25 - 174 CR',
        'TAG $25 - 160 CR',
    ],
    '$35': [
        'BP TAG+$35 - 257 CR',
        'PREM TAG+$35 - 257 CR',
        'PREM TAG+$35 - 259 CR',
        'TAG $35 - 245 CR',
    ],
    '$50': [
        'BP TAG+$50 - 377 CR',
        'PREM TAG+$50 - 379 CR',
        'TAG $50 - 365 CR',
    ],
    '$75': [
        'BP TAG+$75 - 577 CR',
        'PREM TAG+$75 - 579 CR',
        'TAG $75 - 565 CR',
    ],
    '$100': [
        'BP TAG+$100 - 802 CR',
        'PREM TAG+$100 - 804 CR',
        'PROMO BP TAG $100 - 1002 CR',
        'TAG $100 - 790 CR',
    ],
    '$129': [
        'BP TAG+$129 - 1107 CR',
        'PREM TAG+$129 - 1109 CR',
        'TAG $129 - 1095 CR',
    ],
}

In [0]:
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction").filter(
    F.col("VisitDateId").between(20250101, (PY_post_window)) |
    F.col("VisitDateId").between(20260101, (post_window))
)

In [0]:
from pyspark.sql import Row

# Bin mapping DataFrame
mapping_data = [
    Row(Item_Description=item, price_bin=bin_name)
    for bin_name, items in tag_bins.items()
    for item in items
]
bin_mapping_df = spark.createDataFrame(mapping_data)

T_game_band = (
    T_trans
    .withColumn("datefull", F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd"))
    .join(
        F.broadcast(T_location.select(
            "LocationID", "Province_CD", "LC_Trade_Brand_Description",
            "LC_Location_Type_Description", "Location_Name"
        )),
        T_location.LocationID == T_trans.LocationId,
        "left"
    )
    .join(
        F.broadcast(T_item.select("itemskey", "Item_Description")),
        T_item.itemskey == T_trans.ItemSkey,
        "left"
    )
    .join(F.broadcast(bin_mapping_df), on="Item_Description", how="left")
    .filter(F.col("LC_Location_Type_Description").like("%Restaurant%"))
    .filter(F.col("LC_Trade_Brand_Description").like("%Playdium%"))
    .filter(F.col("Item_Description").isin(tags))
)


In [0]:
T_periods = (
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .withColumn(
        "Period",
        F.when(F.month("datefull") == 1, F.lit("Jan"))
         .when(F.month("datefull") == 2, F.lit("Feb"))
         .when(F.month("datefull") == 3, F.lit("Mar"))
         .when(F.month("datefull") == 4, F.lit("Apr_Offer"))
    )
    .filter(F.col("Period").isNotNull())
    .withColumn("Year_Period", F.concat(F.col("Year"), F.lit("_"), F.col("Period")))
)

period_order = [
    "2025_Jan","2025_Feb", "2025_Mar", "2025_Apr_Offer",
    "2026_Jan","2026_Feb", "2026_Mar", "2026_Apr_Offer"
]



In [0]:
# 1. Total Sale Count
T_sale_count_by_period = (
    T_periods
    .groupBy("Location_Name", "price_bin")
    .pivot("Year_Period", period_order)
    .agg(F.sum("Quantity"))
    .orderBy("Location_Name", "price_bin")
)
# 2. Total Sale Scene
T_scene_sale_count_by_period = (
    T_periods
    .groupBy("Location_Name", "price_bin")
    .pivot("Year_Period", period_order)
    .agg(F.sum(F.when(F.col("SCENEFLAG") == "Yes", F.col("Quantity")).otherwise(0)))
    .orderBy("Location_Name", "price_bin")
)

# 2. Total Revenue
T_total_revenue_by_period = (
    T_periods
    .groupBy("Location_Name", "price_bin")
    .pivot("Year_Period", period_order)
    .agg(F.sum("Net_Revenue"))
    .orderBy("Location_Name", "price_bin")
)

# 3. Total Scene Revenue
T_scene_revenue_by_period = (
    T_periods
    .groupBy("Location_Name", "price_bin")
    .pivot("Year_Period", period_order)
    .agg(F.sum(F.when(F.col("SCENEFLAG") == "Yes", F.col("Net_Revenue")).otherwise(0)))
    .orderBy("Location_Name", "price_bin")
)

# 4. Distinct Scene Customers
T_distinct_customers_by_period = (
    T_periods
    .groupBy("Location_Name", "price_bin")
    .pivot("Year_Period", period_order)
    .agg(F.countDistinct("CDE_ID"))
    .orderBy("Location_Name", "price_bin")
)

price_bin_order = ['$5', '$10', '$15', '$25', '$35', '$50', '$75', '$100', '$129']

def sort_by_price(df):
    # Build a CASE WHEN expression mapping each bin to its position
    sort_expr = F.create_map(
        *[item for i, bin_name in enumerate(price_bin_order) for item in (F.lit(bin_name), F.lit(i))]
    )
    return (
        df
        .withColumn("_sort_key", sort_expr[F.col("price_bin")])
        .orderBy("Location_Name", "_sort_key")
        .drop("_sort_key")
    )

T_sale_count_by_period = sort_by_price(T_sale_count_by_period)
T_scene_sale_count_by_period = sort_by_price(T_scene_sale_count_by_period)
T_total_revenue_by_period = sort_by_price(T_total_revenue_by_period)
T_scene_revenue_by_period = sort_by_price(T_scene_revenue_by_period)
T_distinct_customers_by_period = sort_by_price(T_distinct_customers_by_period)

In [0]:
T_sale_count_by_period.display()

In [0]:
T_scene_sale_count_by_period.display()

In [0]:
T_total_revenue_by_period.display()

In [0]:
T_scene_revenue_by_period.display()

In [0]:
T_distinct_customers_by_period.display()
